# Notebook 1 — Data Loading & Inspection

**Goals**

1. Load any tabular forecasting dataset.
2. Produce a one-glance health check of the dataset.
3. Compute per-forecast-key statistics (length, date range, target stats).
4. Spot short series, sparse series and outlier series before deeper analysis.


In [1]:
# ──────────────────────────────────────────────────────────────────────────
# COLAB SETUP — run this once at the top of every tutorial notebook.
# It installs plotly + statsmodels and makes the toolkit importable.
# If you are running locally (not in Colab) the !pip line is harmless.
# ──────────────────────────────────────────────────────────────────────────
!pip install -q plotly statsmodels
import sys, os
# If you uploaded forecasting_toolkit.zip to Colab, unzip it once:
#   !unzip -o forecasting_toolkit.zip
# Otherwise place forecasting_toolkit/ next to this notebook.
sys.path.insert(0, os.path.abspath('.'))

import forecasting_toolkit as ft
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import plotly.io as pio
pio.renderers.default = 'colab'   # change to 'notebook' for local Jupyter


In [2]:
# ──────────────────────────────────────────────────────────────────────────
# POINT THIS AT YOUR DATASET — fill in the four lines below.
# Everything in this notebook works for ANY tabular sales/demand dataset
# (M5, Rossmann, Walmart Store Item Demand, custom CSVs, etc.).
#
#   DATA_PATH    – path to your CSV / Parquet file
#   DATE_COL     – name of the timestamp column
#   TARGET_COL   – name of the column you want to forecast
#   KEY_COLS     – list of columns that together identify ONE time series
#   STATIC_COLS  – columns constant within a key (e.g. store_type, category)
#   DYNAMIC_COLS – columns that vary in time within a key (e.g. promo, price)
#   FREQUENCY    – pandas offset alias: 'D','W','MS','H',…
# ──────────────────────────────────────────────────────────────────────────
DATA_PATH    = './datasets/rohlik_kaggle/sales_processed_tft.csv'
DATE_COL     = 'date'
TARGET_COL   = 'sales'
KEY_COLS     = ['unique_id']
STATIC_COLS  = ['warehouse','product_unique_id','name','L1_category_name_en','L2_category_name_en','L3_category_name_en','L4_category_name_en','country']
DYNAMIC_COLS = ['total_orders','sell_price_main','type_0_discount','type_1_discount','type_2_discount','type_3_discount','type_4_discount','type_5_discount','type_6_discount',
                'holiday','shops_closed','winter_school_holidays','school_holidays','weekday','week','month','day','is_month_start','is_month_end','quarter','weekend','days_since_2020']
FREQUENCY    = 'D'

from forecasting_toolkit import data_io
spec = data_io.make_spec(
    date_col=DATE_COL, target_col=TARGET_COL,
    key_cols=KEY_COLS, static_cols=STATIC_COLS,
    dynamic_cols=DYNAMIC_COLS, frequency=FREQUENCY,
)
df = data_io.load_data(DATA_PATH, spec)
print(f'Loaded {len(df):,} rows × {df.shape[1]} columns')
df.head()


Loaded 4,054,440 rows × 38 columns


,unique_id,date,warehouse,total_orders,sales,sell_price_main,availability,type_0_discount,type_1_discount,type_2_discount,...,month,year,prev_year,day,is_month_start,is_month_end,quarter,weekend,days_since_2020,country
0,0,2022-07-18,Budapest_1,5289.0,3.97,710.89,0.09,0.0,0.0,0.00000,...,7,2022,2022,18,False,False,3,0,929,Hungary
1,0,2022-07-19,Budapest_1,5255.0,73.36,710.89,1.00,0.0,0.0,0.00000,...,7,2022,2022,19,False,False,3,0,930,Hungary
2,0,2022-07-20,Budapest_1,5334.0,558.09,710.89,0.96,0.0,0.0,0.45045,...,7,2022,2022,20,False,False,3,0,931,Hungary
3,0,2022-07-21,Budapest_1,5459.0,14.03,710.89,0.06,0.0,0.0,0.45045,...,7,2022,2022,21,False,False,3,0,932,Hungary
4,0,2022-07-22,Budapest_1,5461.0,558.53,710.89,0.97,0.0,0.0,0.45045,...,7,2022,2022,22,False,False,3,0,933,Hungary


## 1.1 First look — shape, dtypes, head / tail

Before any statistics, eyeball the raw data. The two questions to answer
are:

- **Are the columns the right *type*?** (`object` for what should be a
  number is a red flag; floats stored as strings will silently break
  arithmetic.)
- **Does the data look sorted by `(key, date)`?** Most forecasting
  pipelines assume this; the toolkit's `load_data` enforces it for you.


In [3]:
print(f'Shape:  {df.shape}')
print(f'Memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')
print()
print(df.dtypes.to_string())


Shape:  (4054440, 38)
Memory: 2912.3 MB

unique_id                          int64
date                      datetime64[ns]
warehouse                         object
total_orders                     float64
sales                            float64
sell_price_main                  float64
availability                     float64
type_0_discount                  float64
type_1_discount                  float64
type_2_discount                  float64
type_3_discount                  float64
type_4_discount                  float64
type_5_discount                  float64
type_6_discount                  float64
product_unique_id                  int64
name                              object
L1_category_name_en               object
L2_category_name_en               object
L3_category_name_en               object
L4_category_name_en               object
holiday_name                      object
holiday                            int64
shops_closed                       int64
winter_school_ho

In [4]:
df.head(8)


,unique_id,date,warehouse,total_orders,sales,sell_price_main,availability,type_0_discount,type_1_discount,type_2_discount,...,month,year,prev_year,day,is_month_start,is_month_end,quarter,weekend,days_since_2020,country
0,0,2022-07-18,Budapest_1,5289.0,3.97,710.89,0.09,0.0,0.0,0.00000,...,7,2022,2022,18,False,False,3,0,929,Hungary
1,0,2022-07-19,Budapest_1,5255.0,73.36,710.89,1.00,0.0,0.0,0.00000,...,7,2022,2022,19,False,False,3,0,930,Hungary
2,0,2022-07-20,Budapest_1,5334.0,558.09,710.89,0.96,0.0,0.0,0.45045,...,7,2022,2022,20,False,False,3,0,931,Hungary
3,0,2022-07-21,Budapest_1,5459.0,14.03,710.89,0.06,0.0,0.0,0.45045,...,7,2022,2022,21,False,False,3,0,932,Hungary
4,0,2022-07-22,Budapest_1,5461.0,558.53,710.89,0.97,0.0,0.0,0.45045,...,7,2022,2022,22,False,False,3,0,933,Hungary
5,0,2022-07-23,Budapest_1,4957.0,453.38,710.89,1.00,0.0,0.0,0.45045,...,7,2022,2022,23,False,False,3,1,934,Hungary
6,0,2022-07-24,Budapest_1,5029.0,475.53,710.89,1.00,0.0,0.0,0.45045,...,7,2022,2022,24,False,False,3,1,935,Hungary
7,0,2022-07-25,Budapest_1,5374.0,607.79,710.89,1.00,0.0,0.0,0.45045,...,7,2022,2022,25,False,False,3,0,936,Hungary


In [5]:
df.tail(8)


,unique_id,date,warehouse,total_orders,sales,sell_price_main,availability,type_0_discount,type_1_discount,type_2_discount,...,month,year,prev_year,day,is_month_start,is_month_end,quarter,weekend,days_since_2020,country
4054432,5431,2024-06-09,Prague_2,5908.0,8.005595,135.71,0.933333,0.00000,0.0,0.0,...,6,2024,2024,9,False,False,2,1,1621,Czechia
4054433,5431,2024-06-10,Prague_2,6153.0,11.168767,135.71,0.858904,0.39911,0.0,0.0,...,6,2024,2024,10,False,False,2,0,1622,Czechia
4054434,5431,2024-06-11,Prague_2,5951.0,9.392410,135.71,0.881446,0.39911,0.0,0.0,...,6,2024,2024,11,False,False,2,0,1623,Czechia
4054435,5431,2024-06-12,Prague_2,6108.0,8.918471,81.61,0.926118,0.39911,0.0,0.0,...,6,2024,2024,12,False,False,2,0,1624,Czechia
4054436,5431,2024-06-13,Prague_2,6679.0,9.474471,81.61,0.928000,0.39911,0.0,0.0,...,6,2024,2024,13,False,False,2,0,1625,Czechia
4054437,5431,2024-06-14,Prague_2,6696.0,9.937619,136.47,0.926429,0.39911,0.0,0.0,...,6,2024,2024,14,False,False,2,0,1626,Czechia
4054438,5431,2024-06-15,Prague_2,5677.0,7.561412,81.61,0.950824,0.39911,0.0,0.0,...,6,2024,2024,15,False,False,2,1,1627,Czechia
4054439,5431,2024-06-16,Prague_2,6073.0,8.005595,135.71,0.933333,0.39911,0.0,0.0,...,6,2024,2024,16,False,False,2,1,1628,Czechia


## 1.2 One-table column overview

`inspect_data` returns a **one-row-per-column** summary covering dtype,
missing count, missing %, cardinality and a sample value. This is the
single most useful first-look table you'll produce on any dataset.


In [6]:
overview = ft.data_io.inspect_data(df)
overview


,column,dtype,non_null,missing,missing_pct,n_unique,unique_pct,example
0,unique_id,int64,4054440,0,0.00,5390,0.13,0
1,date,datetime64[ns],4054440,0,0.00,1416,0.03,2022-07-18 00:00:00
2,warehouse,object,4054440,0,0.00,7,0.00,Budapest_1
3,total_orders,float64,4054440,0,0.00,7581,0.19,5289.0
4,sales,float64,4054440,0,0.00,151672,3.74,3.97
5,sell_price_main,float64,4054440,0,0.00,35885,0.89,710.89
6,availability,float64,4054440,0,0.00,20522,0.51,0.09
7,type_0_discount,float64,4054440,0,0.00,18028,0.44,0.0
8,type_1_discount,float64,4054440,0,0.00,154,0.00,0.0
9,type_2_discount,float64,4054440,0,0.00,3598,0.09,0.0


**How to read it**

- `missing_pct` > 0 on the **target** column → you'll need imputation
  (Notebook 5).
- `n_unique` very small on a numeric column → it might really be a
  category in disguise.
- `unique_pct` ≈ 100% on a non-key column → likely a row identifier or
  another timestamp; usually safe to drop.


## 1.3 Top-level coverage report

`coverage_report` gives the bird's-eye view: how many rows, how many
forecast keys, what the date span is, and how many rows per key on
average.


In [7]:
report = ft.data_io.coverage_report(df, spec)
for k, v in report.items():
    print(f'  {k:18s} {v}')


  rows               4054440
  n_forecast_keys    5390
  min_date           2020-08-01 00:00:00
  max_date           2024-06-16 00:00:00
  total_days         1416
  avg_rows_per_key   752.22
  memory_mb          2912.33


**Sanity questions to ask**

- Does `total_days` match `avg_rows_per_key` (give or take a few)? If
  not, some series have many missing dates → **time gaps** (Notebook 4).
- Is the date range what you expected? Some datasets quietly include a
  small "future" portion meant for submission/scoring.


## 1.4 Per-forecast-key summary

The most important table in early forecasting EDA. One row per key,
sorted by length descending.


In [8]:
keys = ft.data_io.summarize_keys(df, spec)
print(f'Total forecast keys: {len(keys):,}')
keys.head(10)


Total forecast keys: 5,390


,unique_id,n_obs,date_min,date_max,target_mean,target_std,target_min,target_max,target_sum,n_zeros,n_missing,span_days,zero_pct,missing_pct
0,4755,1416,2020-08-01,2024-06-16,117.569232,50.471329,6.93,365.74,1.664780e+05,0,0,1415,0.00,0.0
1,80,1416,2020-08-01,2024-06-16,483.636138,290.261911,51.12,1657.18,6.848288e+05,0,0,1415,0.00,0.0
2,1997,1416,2020-08-01,2024-06-16,14.773424,5.064734,0.99,32.53,2.091917e+04,0,0,1415,0.00,0.0
3,1998,1416,2020-08-01,2024-06-16,24.357948,10.097126,0.00,81.36,3.449085e+04,3,0,1415,0.21,0.0
4,2125,1416,2020-08-01,2024-06-16,16.152042,7.381509,0.94,55.06,2.287129e+04,0,0,1415,0.00,0.0
5,2602,1416,2020-08-01,2024-06-16,21.328605,13.449549,0.00,93.96,3.020130e+04,10,0,1415,0.71,0.0
6,4523,1416,2020-08-01,2024-06-16,542.289652,143.413538,102.62,1334.88,7.678821e+05,0,0,1415,0.00,0.0
7,4610,1416,2020-08-01,2024-06-16,10.815766,4.731028,0.00,38.46,1.531512e+04,2,0,1415,0.14,0.0
8,2560,1416,2020-08-01,2024-06-16,709.943710,252.949951,97.88,2366.94,1.005280e+06,0,0,1415,0.00,0.0
9,4611,1416,2020-08-01,2024-06-16,18.327214,8.127642,0.00,49.26,2.595133e+04,6,0,1415,0.42,0.0


In [9]:
keys.describe()


,unique_id,n_obs,target_mean,target_std,target_min,target_max,target_sum,n_zeros,n_missing,span_days,zero_pct,missing_pct
count,5390.000000,5390.000000,5390.000000,5390.000000,5390.000000,5390.000000,5.390000e+03,5390.000000,5390.0,5390.000000,5390.000000,5390.0
mean,2715.770686,752.215213,100.016479,56.751231,7.409147,413.947831,8.146831e+04,9.142301,0.0,931.453803,1.222744,0.0
std,1568.004358,500.980804,259.072715,99.239798,32.249673,772.915952,3.466993e+05,54.587503,0.0,474.930577,5.268586,0.0
min,0.000000,7.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.0,6.000000,0.000000,0.0
25%,1359.250000,265.000000,25.480064,15.255766,0.000000,110.470000,1.045196e+04,0.000000,0.0,524.000000,0.000000,0.0
50%,2716.500000,739.000000,49.067335,29.400688,0.000000,220.480000,2.567841e+04,1.000000,0.0,1062.000000,0.080000,0.0
75%,4071.750000,1296.000000,103.054771,61.776651,4.810000,456.722500,6.422277e+04,4.000000,0.0,1415.000000,0.640000,0.0
max,5431.000000,1416.000000,12283.383037,2903.991487,1143.350000,26316.190000,1.734414e+07,994.000000,0.0,1415.000000,100.000000,0.0


### Distribution of series lengths

Models often perform poorly on very short series. Plotting the length
distribution tells you whether to drop, special-case, or upsample
short series.


In [10]:
fig = ft.plotting.plot_distribution(keys, 'n_obs', bins=40,
                                    title='Distribution of series length (n_obs)')
fig.show()


### Distribution of mean target value

Series with very different mean target values often need different
forecasting strategies (or per-series scaling).


In [11]:
fig = ft.plotting.plot_distribution(keys, 'target_mean', bins=40,
                                    title='Distribution of mean target across series')
fig.show()


## 1.5 Picking example keys for visual exploration

Throughout the rest of the tutorial we'll plot a few example series at a
time. Three useful "personas" to pick:

- the **largest** series (high volume, easy to forecast)
- a **median** series (typical case)
- a **sparse / short** series (where things break)


In [12]:
example_keys = pd.concat([
    keys.head(1),                                             # largest by n_obs
    keys.iloc[[len(keys) // 2]],                              # median by n_obs
    keys.sort_values('zero_pct', ascending=False).head(1),    # most zero-heavy
]).reset_index(drop=True)
example_keys[KEY_COLS + ['n_obs', 'target_mean', 'zero_pct']]


,unique_id,n_obs,target_mean,zero_pct
0,4755,1416,117.569232,0.00
1,651,739,36.804835,0.14
2,2802,8,0.000000,100.00


### Plot the example series

`plot_time_series` accepts a list of key tuples or dictionaries. Here we
build dictionaries from the table above — it's the most readable form.


In [13]:
example_specs = example_keys[KEY_COLS].to_dict('records')
fig = ft.plotting.plot_time_series(df, spec, keys=example_specs,
                                   title='Example forecast keys')
fig.show()


## 1.6 Take-aways

| Observation                         | What it tells you                                  |
|-------------------------------------|----------------------------------------------------|
| Many series shorter than ~30 obs    | Cold-start problem; consider hierarchical pooling  |
| `total_days ≠ avg_rows_per_key`     | Time gaps — Notebook 4 will fix this               |
| High `zero_pct` series              | Intermittent demand — Notebook 3 will classify them|
| Right-skewed target distribution    | Consider log/Box-Cox transform — Notebook 6        |
| Very different `target_mean` scales | Per-series scaling will likely help                |

Next: **Notebook 2 — Exploratory Data Analysis**, where we look at
distributions, autocorrelation and seasonality more carefully.
